# Task 1 — Data Audit & Exploration (CortexHR attrition master table)

**Goal:** profile `attrition_dataset` before any modelling: missing values, outliers, class imbalance and data-leakage risk — and write down a decision for every problem found.

**How to re-run:** `docker compose run --rm audit` (headless) or open this notebook in JupyterLab (`docker compose up jupyter`) and *Run All*. Every number in the written report comes from `reports/findings.json`, which this notebook writes at the end.

**Rule for this notebook:** the raw table is never modified. Everything is measured on `df`, a copy.

| # | Section |
|---|---------|
| 1 | Load & shape |
| 2 | Data dictionary |
| 3 | Duplicates & identity |
| 4 | Missing values |
| 5 | Impossible / inconsistent values |
| 6 | Outliers (IQR vs z-score) |
| 7 | Class imbalance |
| 8 | Data leakage |
| 9 | Check against the planted-problem answer key |
| 10 | Automatic profile (ydata-profiling) vs my manual audit |
| 11 | Decision log |
| 12 | Open items for the Python team |

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists():      # find the repo root from anywhere
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from cortexhr.paths import WORKBOOK, FIGURES, TABLES, DOCS, REPORTS
from cortexhr.audit import (load_attrition, iqr_mask, z_mask, normalise_location,
                            TARGET, LEAKAGE_COLUMNS, NON_FEATURE_COLUMNS)
from cortexhr.data_dictionary import dictionary_frame

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 60, "display.max_colwidth", 80)
for d in (FIGURES, TABLES, DOCS): d.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, GREY, RED, GREEN = "#2F5D8A", "#E07B39", "#9AA5B1", "#C0392B", "#3B8C5A"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 9.5, "axes.titleweight": "bold"})
F = {}      # findings that the written report reads (saved to reports/findings.json)

## 1. Load & shape
The sheet is read exactly as delivered. `df_raw` is the untouched copy; `df` is what I analyse.

In [2]:
df_raw = load_attrition()
df = df_raw.copy()

F["rows"], F["cols"] = df.shape
F["rows_real"] = int((df.source == "master_employee_snapshot").sum())
F["rows_hist"] = int((df.source == "historical_extension").sum())
print(f"{F['rows']:,} rows x {F['cols']} columns")
print(df.source.value_counts().to_string())
print()
print("Column types:", df.dtypes.astype(str).value_counts().to_dict())
print("Observation dates:", df.observation_date.min().date(), "->", df.observation_date.max().date())
print("Real snapshot rows all on:", df.loc[df.source == 'master_employee_snapshot', 'observation_date'].dt.date.unique())
df.head(3).T

2,389 rows x 42 columns
source
historical_extension        2000
master_employee_snapshot     389

Column types: {'float64': 15, 'int64': 14, 'object': 10, 'datetime64[ns]': 3}
Observation dates: 2021-01-01 -> 2025-09-30
Real snapshot rows all on: [datetime.date(2025, 9, 30)]


,0,1,2
row_id,1,2,3
employee_id,EMP-0002,EMP-0004,EMP-0011
observation_date,2025-09-30 00:00:00,2025-09-30 00:00:00,2025-09-30 00:00:00
department_id,D08,D03,D07
role_id,R047,R017,R039
job_level,4,3,1
work_location,Pune,Erode,Kochi
work_mode,Remote,Hybrid,Hybrid
employment_type,Permanent,Permanent,Permanent
highest_education,B.E. Computer Science,MCA,M.A. Social Work


**What I see.** 2,389 rows × 42 columns. Two kinds of rows are mixed together: 389 *real* employees measured on 30 Sep 2025 (`EMP-…`) and 2,000 *simulated historical* rows measured on dates between 2021 and 2025 (`HIST-…`). The README says 383 real employees, so 6 rows are already suspicious (section 3).

## 2. Data dictionary (in my own words)
Each column gets a *role*: `FEATURE` (safe model input), `LABEL` (what we predict), `LEAK` (only known after the outcome — see section 8), `ID`, or `PROVENANCE` (describes the data, not the person). The facts (missing %, distinct values) are measured, not copied.

In [3]:
dd = dictionary_frame(df_raw)
dd.to_csv(TABLES / "data_dictionary.csv", index=False)

lines = ["# Data dictionary — `attrition_dataset`", "",
         "Written for Task 1. Roles: FEATURE = safe model input, LABEL = target, LEAK = only known after the outcome,",
         "ID = identifier, PROVENANCE = describes where the row came from.", "",
         "| Column | Role | Type | Missing | Meaning | Audit note |", "|---|---|---|---|---|---|"]
for r in dd.itertuples():
    lines.append(f"| `{r.column}` | {r.role} | {r.kind} | {r.missing_pct}% | {r.meaning} | {r.audit_note} |")
(DOCS / "data_dictionary.md").write_text("\n".join(lines) + "\n")

F["role_counts"] = dd.role.value_counts().to_dict()
print(dd.role.value_counts().to_string())
dd[["column", "role", "kind", "missing_pct", "distinct", "meaning"]]

role
FEATURE       31
LEAK           6
ID             2
PROVENANCE     2
LABEL          1


,column,role,kind,missing_pct,distinct,meaning
0,row_id,ID,id,0.0,2389,Running row number in the sheet.
1,employee_id,ID,id,0.0,2383,"Employee code. EMP-xxxx = real employee, HIST-xxxx = simulated history."
2,observation_date,PROVENANCE,date,0.0,1135,"The day the features were measured (30 Sep 2025 for real staff, 2021-2025 fo..."
3,department_id,FEATURE,category,0.0,10,Department code D00-D10 (D01 Engineering ... D10 Legal).
4,role_id,FEATURE,category,0.0,48,Job role code (48 distinct roles).
5,job_level,FEATURE,count,0.0,6,Seniority level 1 (associate) to 6 (head).
6,work_location,FEATURE,category,0.0,16,Office city.
7,work_mode,FEATURE,category,0.0,3,"Onsite, Hybrid or Remote."
8,employment_type,FEATURE,category,0.0,2,Permanent or fixed-term contract.
9,highest_education,FEATURE,category,0.6,27,"Highest qualification, 27 free-text values."


## 3. Duplicates & identity
A duplicate is dangerous: if one copy lands in the training set and the other in the test set, the model is tested on something it has already seen, and the score looks better than it is.

In [4]:
content_cols = [c for c in df.columns if c != "row_id"]          # row_id is just a counter, ignore it
dup_mask = df.duplicated(subset=content_cols, keep="first")      # True for the 2nd+ copy
F["dup_rows"] = int(dup_mask.sum())
F["dup_employee_ids"] = sorted(df.loc[dup_mask, "employee_id"])
F["unique_employees"] = int(df.employee_id.nunique())
F["real_unique_employees"] = int(df.loc[df.source == "master_employee_snapshot", "employee_id"].nunique())

print("Exact duplicate rows (ignoring row_id):", F["dup_rows"])
print("Unique employee_ids:", F["unique_employees"], "of", len(df), "rows")
print("Real employees after de-duplication:", F["real_unique_employees"], "(README says 383)")
display(df[df.employee_id.isin(F["dup_employee_ids"])].sort_values(["employee_id", "row_id"])
          [["row_id", "employee_id", "observation_date", "annual_fixed_pay", TARGET]])
print("Duplicates all sit at the END of the sheet? row_ids:", df.loc[dup_mask, "row_id"].tolist())

Exact duplicate rows (ignoring row_id): 6
Unique employee_ids: 2383 of 2389 rows
Real employees after de-duplication: 383 (README says 383)


,row_id,employee_id,observation_date,annual_fixed_pay,left_within_12m
26,27,EMP-0060,2025-09-30,1464708,0
2386,2387,EMP-0060,2025-09-30,1464708,0
100,101,EMP-0237,2025-09-30,486732,0
2387,2388,EMP-0237,2025-09-30,486732,0
101,102,EMP-0238,2025-09-30,795984,0
2384,2385,EMP-0238,2025-09-30,795984,0
150,151,EMP-0325,2025-09-30,3157536,0
2383,2384,EMP-0325,2025-09-30,3157536,0
250,251,EMP-0483,2025-09-30,1598520,0
2385,2386,EMP-0483,2025-09-30,1598520,0


Duplicates all sit at the END of the sheet? row_ids: [2384, 2385, 2386, 2387, 2388, 2389]


**Finding.** Six employees appear twice, and the copies are identical in every column. The copies are the last six rows of the sheet. After removing them, 383 real employees remain, which matches the README.
**Decision:** drop the second copy of each. In Task 2, split data by `employee_id` (grouped splitting) so a person can never straddle train and test.

## 4. Missing values
Two questions for each column: **how much** is missing, and **why**. The *why* decides the fix — missing because it doesn't apply is very different from missing because someone forgot to record it.

In [5]:
miss = pd.DataFrame({"missing_n": df.isna().sum()})
miss["missing_pct"] = (100 * miss.missing_n / len(df)).round(1)
miss = miss[miss.missing_n > 0].sort_values("missing_pct", ascending=False)
miss

,missing_n,missing_pct
resignation_date,2068,86.6
exit_date,2040,85.4
exit_reason,2040,85.4
fnf_settlement_amount,2040,85.4
exit_interview_done,2040,85.4
months_since_last_promotion,1459,61.1
last_engagement_score,528,22.1
last_raise_pct,202,8.5
last_performance_rating,182,7.6
commute_distance_km,55,2.3


In [6]:
# WHY is each one missing?  Test the obvious explanation for every column.
print("months_since_last_promotion blank  <->  promotions_received == 0 :")
display(pd.crosstab(df.months_since_last_promotion.isna(), df.promotions_received == 0,
                    rownames=["promotion months blank"], colnames=["never promoted"]))
print("last_raise_pct blank  <->  raises_received == 0 :")
display(pd.crosstab(df.last_raise_pct.isna(), df.raises_received == 0,
                    rownames=["raise % blank"], colnames=["never had a raise"]))
print("last_performance_rating blank, by tenure < 12 months:")
display(pd.crosstab(df.last_performance_rating.isna(), df.tenure_months < 12,
                    rownames=["rating blank"], colnames=["tenure < 12m"]))
print("Exit columns blank  <->  employee stayed:")
display(pd.crosstab(df.exit_date.isna(), df[TARGET], rownames=["exit_date blank"], colnames=[TARGET]))

months_since_last_promotion blank  <->  promotions_received == 0 :


never promoted,False,True
promotion months blank,,
False,930,0
True,0,1459


last_raise_pct blank  <->  raises_received == 0 :


never had a raise,False,True
raise % blank,,
False,2187,0
True,0,202


last_performance_rating blank, by tenure < 12 months:


tenure < 12m,False,True
rating blank,,
False,2084,123
True,0,182


Exit columns blank  <->  employee stayed:


left_within_12m,0,1
exit_date blank,,
False,0,349
True,2040,0


In [7]:
# Is the 'missingness' itself informative?  Compare attrition rate for rows with / without a value.
rows = []
for c in ["last_engagement_score", "last_performance_rating", "commute_distance_km", "highest_education",
          "last_raise_pct", "months_since_last_promotion"]:
    m = df[c].isna()
    ct = pd.crosstab(m, df[TARGET])
    p = stats.chi2_contingency(ct)[1]
    rows.append({"column": c, "attrition_if_missing_%": round(100 * df.loc[m, TARGET].mean(), 1),
                 "attrition_if_present_%": round(100 * df.loc[~m, TARGET].mean(), 1),
                 "n_missing": int(m.sum()), "chi2_p_value": round(p, 3)})
miss_signal = pd.DataFrame(rows)
display(miss_signal)
print("Engagement blanks by source:", df.groupby('source').last_engagement_score.apply(lambda s: round(100*s.isna().mean(),1)).to_dict())

,column,attrition_if_missing_%,attrition_if_present_%,n_missing,chi2_p_value
0,last_engagement_score,16.7,14.0,528,0.148
1,last_performance_rating,15.4,14.5,182,0.842
2,commute_distance_km,9.1,14.7,55,0.328
3,highest_education,13.3,14.6,15,1.000
4,last_raise_pct,16.3,14.4,202,0.534
5,months_since_last_promotion,15.1,13.8,1459,0.382


Engagement blanks by source: {'historical_extension': 23.8, 'master_employee_snapshot': 13.4}


In [8]:
# Plan for each column: cause, plan, and the figure
cause = {
    "months_since_last_promotion": ("Structural: never promoted", "KEEP + flag + fill with tenure_months"),
    "last_raise_pct":              ("Structural: never had a raise", "KEEP + flag + fill with 0"),
    "last_performance_rating":     ("Structural: joined <12 months ago, no appraisal yet", "KEEP + flag + fill with median"),
    "last_engagement_score":       ("Non-response (22%)", "KEEP + flag + fill with median"),
    "commute_distance_km":         ("Random gaps (2.3%)", "KEEP + flag + fill with median by city"),
    "highest_education":           ("Random gaps (0.6%)", "KEEP + fill with 'Unknown'"),
}
for c in LEAKAGE_COLUMNS[:5]:
    cause[c] = ("Exit information: blank for everyone who stayed", "DROP (leakage, see section 8)")
miss["cause"] = [cause[c][0] for c in miss.index]
miss["plan"] = [cause[c][1] for c in miss.index]
miss.to_csv(TABLES / "missing_values_plan.csv")
F["missing"] = miss.reset_index().rename(columns={"index": "column"}).to_dict("records")
F["missing_signal"] = miss_signal.to_dict("records")
display(miss)

colors = {"Structural": BLUE, "Non-response": ORANGE, "Random": GREEN, "Exit": GREY}
def col_of(txt): return next(v for k, v in colors.items() if txt.startswith(k))
fig, ax = plt.subplots(figsize=(7.2, 3.3))
m = miss.sort_values("missing_pct")
ax.barh(m.index, m.missing_pct, color=[col_of(t) for t in m.cause])
for y, v in enumerate(m.missing_pct): ax.text(v + 1, y, f"{v}%", va="center", fontsize=8.5)
ax.set_xlim(0, 100); ax.set_xlabel("% of rows missing"); ax.set_title("Missing values by column, coloured by cause")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label="Structural (doesn't apply)"), Patch(color=ORANGE, label="Non-response"),
                   Patch(color=GREEN, label="Random gaps"), Patch(color=GREY, label="Exit info (leavers only)")],
          loc="lower right", frameon=False, fontsize=8)
ax.grid(axis="y", visible=False); plt.tight_layout(); plt.savefig(FIGURES / "fig1_missing.png"); plt.show()

,missing_n,missing_pct,cause,plan
resignation_date,2068,86.6,Exit information: blank for everyone who stayed,"DROP (leakage, see section 8)"
exit_date,2040,85.4,Exit information: blank for everyone who stayed,"DROP (leakage, see section 8)"
exit_reason,2040,85.4,Exit information: blank for everyone who stayed,"DROP (leakage, see section 8)"
fnf_settlement_amount,2040,85.4,Exit information: blank for everyone who stayed,"DROP (leakage, see section 8)"
exit_interview_done,2040,85.4,Exit information: blank for everyone who stayed,"DROP (leakage, see section 8)"
months_since_last_promotion,1459,61.1,Structural: never promoted,KEEP + flag + fill with tenure_months
last_engagement_score,528,22.1,Non-response (22%),KEEP + flag + fill with median
last_raise_pct,202,8.5,Structural: never had a raise,KEEP + flag + fill with 0
last_performance_rating,182,7.6,"Structural: joined <12 months ago, no appraisal yet",KEEP + flag + fill with median
commute_distance_km,55,2.3,Random gaps (2.3%),KEEP + flag + fill with median by city


**Findings.**
* Every blank in the four "career" columns has an explanation — **none of it is random**. A blank promotion date means *never promoted*; a blank raise means *never had a raise*; a blank rating means *joined under a year ago* (all 182 blanks have tenure < 12 months).
* Engagement is blank for 22% of people (no survey response). Their attrition rate (16.7%) is a little higher than for responders (14.0%), but the gap is **not statistically significant** (p ≈ 0.15), so I flag it and let the model decide.
* Commute (55 rows) and education (15 rows) are small, random-looking gaps.
* The five exit columns are blank for exactly the people who stayed, which is a leakage signal rather than a missing-data problem.

### 🧠 "A column has 40% missing values — drop, fill or flag?" — my answer
**I don't decide from the percentage; I decide from the reason.**
1. *Why is it missing?* If the blank carries meaning ("never promoted"), the blank **is** information. Dropping the column throws it away.
2. *Is the column useful when present?* If yes, keep it.
3. *Then:* **fill** with a sensible neutral value so models can run **and add a "was missing" flag** so the model can still see the difference. Filling alone hides the fact that the value was never real (a never-promoted person doesn't *really* have 0 months since promotion).
4. **Drop** only when the column is mostly blank *and* the blank has no meaning *and* the value adds no signal — or when it is a leakage column.

The data has a real example: `months_since_last_promotion` is 61% blank, higher than 40%, yet it stays: the blank means "never promoted", which is a strong, legitimate signal. Fill rules are learned from the **training split only** and applied to the test split, otherwise test information leaks into training.

## 5. Impossible and inconsistent values
Outliers are *unusual but possible*. These are values that **cannot be true**. I wrote explicit rules from the data dictionary and counted violations.

In [9]:
exp_gap = df.total_experience_years - df.tenure_months / 12      # total experience must be >= time spent here
loc_clean = df.work_location.map(normalise_location)

rules = {
    "tenure_months < 0":                                   df.tenure_months < 0,
    "attendance_rate_3m > 1 (more than 100%)":             df.attendance_rate_3m > 1,
    "wfh_share_3m < 0":                                    df.wfh_share_3m < 0,
    "wfh_share_3m > 1":                                    df.wfh_share_3m > 1,
    "total_experience_years < 0":                          df.total_experience_years < 0,
    "total_experience < tenure (by more than 0.1 yr)":    exp_gap < -0.1,
    "last_performance_rating outside 1-5":                 ~df.last_performance_rating.dropna().between(1, 5).reindex(df.index, fill_value=True),
    "last_engagement_score outside 1-5":                   ~df.last_engagement_score.dropna().between(1, 5).reindex(df.index, fill_value=True),
    "job_level outside 1-6":                               ~df.job_level.between(1, 6),
    "comp_ratio <= 0":                                     df.comp_ratio <= 0,
    "work_location spelled inconsistently":                df.work_location != loc_clean,
    "promotions_received=0 but months_since_promotion given": (df.promotions_received == 0) & df.months_since_last_promotion.notna(),
    "months_since_last_promotion > tenure":               df.months_since_last_promotion > df.tenure_months + 1,
}
viol = pd.DataFrame({"rule": list(rules), "rows_violating": [int(m.sum()) for m in rules.values()],
                     "in_real_snapshot": [int((m & (df.source == 'master_employee_snapshot')).sum()) for m in rules.values()],
                     "in_historical": [int((m & (df.source == 'historical_extension')).sum()) for m in rules.values()]})
viol.to_csv(TABLES / "impossible_values.csv", index=False)
F["violations"] = viol.to_dict("records")
display(viol)
print("Spelling variants found:", sorted(df.loc[df.work_location != loc_clean, "work_location"].unique()))
print("Locations: raw =", df.work_location.nunique(), "-> cleaned =", loc_clean.nunique())
print("\nNegative-tenure rows:"); display(df.loc[df.tenure_months < 0, ["employee_id", "tenure_months", "total_experience_years", "source"]])
print("wfh_share out of range, by work_mode:"); display(df[(df.wfh_share_3m < 0) | (df.wfh_share_3m > 1)].groupby("work_mode").wfh_share_3m.agg(["count", "min", "max"]))
print("Experience gap (total_experience - tenure in yrs) where violated:"); display(exp_gap[exp_gap < -0.1].describe().round(2))

,rule,rows_violating,in_real_snapshot,in_historical
0,tenure_months < 0,4,4,0
1,attendance_rate_3m > 1 (more than 100%),2,0,2
2,wfh_share_3m < 0,190,0,190
3,wfh_share_3m > 1,23,0,23
4,total_experience_years < 0,10,0,10
5,total_experience < tenure (by more than 0.1 yr),137,0,137
6,last_performance_rating outside 1-5,0,0,0
7,last_engagement_score outside 1-5,0,0,0
8,job_level outside 1-6,0,0,0
9,comp_ratio <= 0,0,0,0


Spelling variants found: ['BENGALURU', 'Bangalore', 'Delhi-NCR', 'Madras', 'bengaluru', 'chennai', 'delhi ncr']
Locations: raw = 16 -> cleaned = 9

Negative-tenure rows:


,employee_id,tenure_months,total_experience_years,source
108,EMP-0248,-4,8.0,master_employee_snapshot
125,EMP-0280,-5,11.7,master_employee_snapshot
312,EMP-0568,-2,1.1,master_employee_snapshot
339,EMP-0596,-1,5.9,master_employee_snapshot


wfh_share out of range, by work_mode:


,count,min,max
work_mode,,,
Onsite,190,-0.096,-0.001
Remote,23,1.003,1.088


Experience gap (total_experience - tenure in yrs) where violated:


count    137.00
mean      -0.90
std        0.63
min       -2.72
25%       -1.20
50%       -0.80
75%       -0.42
max       -0.12
dtype: float64

**Findings.**
* **4 rows have negative tenure** (the observation date is before the join date): EMP-0248, EMP-0280, EMP-0568, EMP-0596. All are real employees. **I have not corrected them**: the true join dates are not in the workbook, so this is open item O1 (section 12).
* **2 rows have attendance above 100%** (1.12).
* **13 rows spell a city differently** (`BENGALURU`, `Bangalore`, `Madras`, `Delhi-NCR` …). Left alone, a model would treat "Bangalore" and "Bengaluru" as two different cities (16 labels instead of 9).
* **Extra findings beyond the obvious checks:** `wfh_share_3m` strays outside 0–1 for ~213 rows (tiny negatives for onsite staff, slightly above 1 for remote staff — noise added during data generation), and **137 historical rows claim fewer total years of experience than years at the company** (up to 2.7 years short), which is impossible given the definition. These occur **only** in the simulated historical rows, hinting that the two sources were generated by slightly different logic.

**Decisions:** fix spelling with a lookup table; cap attendance at 1; clip shares to 0–1; floor total experience at tenure. **Negative tenure is deliberately NOT corrected:** it waits for the Python team's join dates (open item O1, section 12). The other rules go into Task 2's pipeline.

## 6. Outliers: IQR vs z-score
Two standard tests on salary, working hours and tenure (plus overtime and commute as extra checks).
* **IQR (Tukey) rule:** flag values more than 1.5 × IQR beyond the 25th/75th percentile. Based on percentiles, so a few extreme values don't distort it.
* **z-score rule:** flag values more than 3 standard deviations from the mean. Based on mean and standard deviation, which extreme values themselves pull around.

In [10]:
d = df.drop_duplicates(subset=content_cols).copy()         # analyse each person once
targets = {"annual_fixed_pay": "Salary (INR/yr)", "avg_hours_worked_3m": "Daily hours worked",
           "tenure_months": "Tenure (months)", "avg_overtime_hours_per_month_3m": "Overtime hrs/month", "commute_distance_km": "Commute (km)"}
rows = []
for c, label in targets.items():
    s = d[c].dropna(); a, b = iqr_mask(s), z_mask(s)
    rows.append({"variable": label, "column": c, "n": len(s), "skewness": round(stats.skew(s), 2),
                 "IQR_flags": int(a.sum()), "z_flags": int(b.sum()), "both": int((a & b).sum()),
                 "IQR_only": int((a & ~b).sum()), "z_only": int((b & ~a).sum()),
                 "min": round(s.min(), 2), "median": round(s.median(), 2), "max": round(s.max(), 2),
                 "IQR_lower": round(s.quantile(.25) - 1.5 * (s.quantile(.75) - s.quantile(.25)), 2),
                 "IQR_upper": round(s.quantile(.75) + 1.5 * (s.quantile(.75) - s.quantile(.25)), 2),
                 "z_lower": round(s.mean() - 3 * s.std(), 2), "z_upper": round(s.mean() + 3 * s.std(), 2)})
outl = pd.DataFrame(rows); outl.to_csv(TABLES / "outlier_methods.csv", index=False)
F["outliers"] = outl.to_dict("records")
display(outl.drop(columns="column"))
print("Share of overtime values that are exactly zero:", round(100 * (d.avg_overtime_hours_per_month_3m == 0).mean(), 1), "%")

,variable,n,skewness,IQR_flags,z_flags,both,IQR_only,z_only,min,median,max,IQR_lower,IQR_upper,z_lower,z_upper
0,Salary (INR/yr),2383,4.81,93,26,26,67,0,148848.0,756972.00,18671760.00,-990153.00,2838351.0,-1541764.76,3597964.47
1,Daily hours worked,2383,-4.13,48,15,15,33,0,1.8,8.68,9.48,7.93,9.4,7.65,9.66
2,Tenure (months),2383,1.40,73,45,45,28,0,-5.0,31.00,164.00,-31.50,100.5,-44.48,119.99
3,Overtime hrs/month,2383,2.97,445,61,61,384,0,0.0,0.00,50.00,0.00,0.0,-13.84,18.18
4,Commute (km),2328,2.02,141,47,47,94,0,1.3,9.70,65.00,-7.90,29.7,-15.69,40.52


Share of overtime values that are exactly zero: 81.3 %


**How the two methods compare.**
* Every z-score flag is also an IQR flag, but **IQR flags between 1.6× and 7× more rows** (e.g. 93 vs 26 for salary; 445 vs 61 for overtime).
* Why: salary, tenure and overtime are strongly right-skewed. The few huge values inflate the standard deviation, so the z-score "fence" sits far out and misses moderately high values. The IQR fence is closer in.
* Overtime shows IQR's weak spot: over 80% of people have **zero** overtime, so the IQR is 0 and *any* overtime at all gets flagged (445 rows). Those are normal overtime-eligible staff, not errors.
* Neither method can tell an *error* from a *legitimately big value*. A 10-year veteran or a senior leader is statistically unusual but real. I need a **business rule** for that, below.

In [11]:
# Salary deep-dive.  Raw salary is dominated by seniority, so a global z-score is the wrong lens.
# Business-rule check: pay / comp_ratio should equal the role's mid-point.  If it is wildly different from other people in
# the SAME role, the pay figure is probably mistyped.
d["implied_mid"] = d.annual_fixed_pay / d.comp_ratio
d["mid_vs_role_median"] = d.implied_mid / d.groupby("role_id").implied_mid.transform("median")
print("Spread of the mid-point check (should hug 1.0):"); display(d.mid_vs_role_median.describe().round(3))

typos = d[d.mid_vs_role_median > 3][["employee_id", "role_id", "job_level", "annual_fixed_pay", "comp_ratio", "mid_vs_role_median"]].copy()
typos["pay_if_divided_by_10"] = typos.annual_fixed_pay / 10
typos["level_median_pay"] = typos.job_level.map(d.groupby("job_level").annual_fixed_pay.median())
display(typos)
F["salary_typos"] = typos[["employee_id", "annual_fixed_pay", "pay_if_divided_by_10", "job_level"]].to_dict("records")

# Compare the three methods on the SAME salary column
s = d.annual_fixed_pay
within = d.groupby("job_level").annual_fixed_pay.transform(lambda x: (x - x.mean()) / x.std()).abs() > 3
F["salary_method_counts"] = {"iqr_raw": int(iqr_mask(s).sum()), "z_raw": int(z_mask(s).sum()),
                             "z_log": int(z_mask(np.log(s)).sum()), "z_within_level": int(within.sum()),
                             "business_rule_mid_ratio": int((d.mid_vs_role_median > 3).sum())}
print(F["salary_method_counts"])

Spread of the mid-point check (should hug 1.0):


count    2383.000
mean        1.017
std         0.371
min         0.820
25%         0.936
50%         1.000
75%         1.069
max        11.366
Name: mid_vs_role_median, dtype: float64

,employee_id,role_id,job_level,annual_fixed_pay,comp_ratio,mid_vs_role_median,pay_if_divided_by_10,level_median_pay
14,EMP-0034,R015,1,4888200,0.966,11.365549,488820.0,359664.0
216,EMP-0433,R006,3,18671760,1.279,11.144261,1867176.0,1196676.0
344,EMP-0601,R034,2,3606000,0.975,11.112915,360600.0,653598.0


{'iqr_raw': 93, 'z_raw': 26, 'z_log': 1, 'z_within_level': 5, 'business_rule_mid_ratio': 3}


In [12]:
# Figure 2:  (a) how many rows each method flags,  (b) salary by level with the 3 typos highlighted
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.2, 3.5), gridspec_kw={"width_ratios": [1, 1.1]})
o = outl[outl.column.isin(["annual_fixed_pay", "avg_hours_worked_3m", "tenure_months"])]
x = np.arange(len(o)); w = .38
b1 = a1.bar(x - w/2, o.IQR_flags, w, color=BLUE, label="IQR rule"); b2 = a1.bar(x + w/2, o.z_flags, w, color=ORANGE, label="z-score > 3")
a1.bar_label(b1, fontsize=8.5); a1.bar_label(b2, fontsize=8.5)
a1.set_xticks(x); a1.set_xticklabels(["Salary", "Daily hours", "Tenure"]); a1.set_ylabel("rows flagged (of 2,383)")
a1.set_ylim(0, 108); a1.set_title("(a) IQR flags more rows than z-score"); a1.legend(frameon=False); a1.grid(axis="x", visible=False)

rng = np.random.default_rng(1); jit = rng.uniform(-.25, .25, len(d))
isty = d.mid_vs_role_median > 3
a2.scatter(d.job_level[~isty] + jit[~isty], d.annual_fixed_pay[~isty], s=5, color=GREY, alpha=.5, label="normal")
a2.scatter(d.job_level[isty] + jit[isty], d.annual_fixed_pay[isty], s=45, color=RED, label="10x typo (3 rows)", zorder=5)
a2.set_yscale("log"); a2.set_xlabel("Job level"); a2.set_ylabel("Annual fixed pay (INR, log scale)")
a2.set_title("(b) Three salaries are 10x too high"); a2.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(FIGURES / "fig2_outliers.png"); plt.show()

**Salary.** The IQR rule flags 93 salaries and z-score 26, but most of those are just senior people with big (genuine) pay. The business-rule check — *does pay ÷ comp_ratio match the role's midpoint?* — isolates **exactly three** rows that are ~11× too high (EMP-0034, EMP-0433, EMP-0601). Dividing by 10 puts each of them right where its job level belongs, so the cause is a typo (an extra zero), not a real salary.
**Decision:** correct the three typos (÷10); *keep* the genuine high earners; use a log transform for pay in linear models.

In [13]:
# Hours worked and tenure: look at the extremes
print("Lowest daily-hours values:")
display(d.nsmallest(5, "avg_hours_worked_3m")[["employee_id", "avg_hours_worked_3m", "attendance_rate_3m", "unpaid_leave_days_6m", "paid_leave_days_6m", TARGET]])
print("Highest overtime values:")
display(d.nlargest(5, "avg_overtime_hours_per_month_3m")[["employee_id", "job_level", "avg_overtime_hours_per_month_3m", "avg_hours_worked_3m"]])
print("Longest tenures (months):"); print(d.tenure_months.nlargest(5).tolist(), "| median:", d.tenure_months.median())
F["hours_extreme"] = d.nsmallest(1, "avg_hours_worked_3m")[["employee_id", "avg_hours_worked_3m", "attendance_rate_3m"]].to_dict("records")[0]

Lowest daily-hours values:


,employee_id,avg_hours_worked_3m,attendance_rate_3m,unpaid_leave_days_6m,paid_leave_days_6m,left_within_12m
143,EMP-0313,1.80,0.2031,1.0,51,0
317,EMP-0573,6.65,0.7656,4.0,16,0
8,EMP-0025,7.00,0.7969,1.0,15,0
212,EMP-0428,7.20,0.8281,1.5,13,0
283,EMP-0533,7.23,0.8281,2.0,11,0


Highest overtime values:


,employee_id,job_level,avg_overtime_hours_per_month_3m,avg_hours_worked_3m
1901,HIST-1519,1,50.00,9.48
745,HIST-0363,2,36.24,9.12
2382,HIST-2000,1,33.89,9.45
736,HIST-0354,1,31.11,9.20
851,HIST-0469,2,30.75,8.53


Longest tenures (months):
[164, 162, 160, 160, 160] | median: 31.0


**Hours.** One person (EMP-0313) works 1.8 h/day with 20% attendance — far below everybody else (next lowest: 6.65 h). The two numbers agree with each other, so it looks like a genuine long absence rather than a typo. **Decision:** keep, flag for HR to confirm.
**Overtime** (max 50 h/month) is rare but plausible for overtime-eligible roles. **Decision:** keep, transform with `log1p`.
**Tenure.** The 4 negative values are errors (section 5). The long tail up to 164 months (13.7 years) is real. **Decision:** keep.

## 7. Class imbalance
How many people left within 12 months?

In [14]:
y = d[TARGET]
F["label_counts"] = {"stayed": int((y == 0).sum()), "left": int((y == 1).sum())}
F["label_rate_pct"] = round(100 * y.mean(), 1)
F["dummy_accuracy_pct"] = round(100 * (1 - y.mean()), 1)
F["imbalance_ratio"] = round((y == 0).sum() / (y == 1).sum(), 1)
print(F["label_counts"], f"-> {F['label_rate_pct']}% left. Ratio {F['imbalance_ratio']}:1")
print(f"A model that always says 'stays' is {F['dummy_accuracy_pct']}% accurate and finds nobody.")

by_src = d.groupby("source")[TARGET].agg(rate="mean", n="count", left="sum"); by_src["rate"] = (100 * by_src.rate).round(1)
by_year = d.groupby(d.observation_date.dt.year)[TARGET].agg(rate="mean", n="count"); by_year["rate"] = (100 * by_year.rate).round(1)
dept_names = pd.read_excel(WORKBOOK, "monthly_headcount_history").drop_duplicates("department_id").set_index("department_id").department_name
by_dept = d.groupby("department_id")[TARGET].agg(rate="mean", n="count"); by_dept["rate"] = (100 * by_dept.rate).round(1)
by_dept["name"] = by_dept.index.map(dept_names); by_dept = by_dept.sort_values("rate")
F["by_source"] = by_src.reset_index().to_dict("records"); F["by_year"] = by_year.reset_index().to_dict("records")
F["by_dept"] = by_dept.reset_index().to_dict("records")
display(by_src); display(by_year.T); display(by_dept[["name", "n", "rate"]].T)
print("Voluntary vs involuntary among leavers:", d.loc[d[TARGET] == 1, "voluntary_left_within_12m"].value_counts().to_dict())

{'stayed': 2034, 'left': 349} -> 14.6% left. Ratio 5.8:1
A model that always says 'stays' is 85.4% accurate and finds nobody.


,rate,n,left
source,,,
historical_extension,15.0,2000,299
master_employee_snapshot,13.1,383,50


observation_date,2021,2022,2023,2024,2025
rate,15.0,13.5,14.3,16.3,14.0
n,514.0,467.0,446.0,471.0,485.0


department_id,D10,D07,D05,D03,D08,D01,D04,D02,D09,D06
name,Legal & Compliance,Human Resources,Marketing,Data & AI,Finance & Accounts,Engineering,Sales,Product & Design,Operations & IT,Customer Support
n,62,81,102,243,111,806,266,161,228,323
rate,6.5,11.1,11.8,13.2,13.5,13.6,13.9,14.3,17.1,21.1


Voluntary vs involuntary among leavers: {1: 321, 0: 28}


In [15]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.2, 3.3), gridspec_kw={"width_ratios": [.8, 1.4]})
vals = [F["label_counts"]["stayed"], F["label_counts"]["left"]]
bars = a1.bar(["Stayed", "Left within\n12 months"], vals, color=[BLUE, ORANGE], width=.6)
for b, v in zip(bars, vals): a1.text(b.get_x() + b.get_width()/2, v + 30, f"{v:,}\n({100*v/sum(vals):.1f}%)", ha="center", fontsize=9)
a1.set_ylim(0, 2500); a1.set_ylabel("employees"); a1.set_title("(a) About 6 stay for every 1 who leaves"); a1.grid(axis="x", visible=False)
bd = by_dept
a2.barh(bd.name, bd.rate, color=BLUE); a2.axvline(F["label_rate_pct"], color=ORANGE, ls="--", lw=1.5)
a2.text(F["label_rate_pct"] + .3, -0.75, f"overall {F['label_rate_pct']}%", color=ORANGE, fontsize=8.5)
for i, (r, n) in enumerate(zip(bd.rate, bd.n)): a2.text(r + .3 if r > 16 else 15.4, i, f"{r}%  (n={n})", va="center", fontsize=8)   # keep labels off the dashed line
a2.set_xlim(0, 30); a2.set_xlabel("% who left within 12 months"); a2.set_title("(b) Attrition rate by department"); a2.grid(axis="y", visible=False)
plt.tight_layout(); plt.savefig(FIGURES / "fig3_imbalance.png"); plt.show()

**Finding.** **14.6% of people left (349 of 2,383); 85.4% stayed** — about 6 stayers per leaver. This is a *moderate* imbalance, not extreme, but it is enough to make **accuracy a misleading score**: a model that predicts "nobody leaves" is 85.4% accurate while being useless. The rate is steady across years (13–16%), so the label looks stable over time; real employees leave at 12.9% versus 15.0% in the simulated rows. Department rates range widely, but with small groups (e.g. only 62 rows in Legal) those differences are noisy.
**Decisions:** report **ROC-AUC, PR-AUC, precision and recall** instead of accuracy; use **stratified** splits so each fold keeps ~14.6% leavers; start with `class_weight="balanced"` rather than copying rows; and if oversampling (e.g. SMOTE) is tried later, do it **inside the training fold only**, never before splitting.

## 8. Data leakage
**Data leakage** = the model sees information at training time that will *not exist* when it has to make a real prediction. The model then looks brilliant in testing and fails in real life.

The prediction moment is **30 Sep 2025** ("will this person leave in the next 12 months?"). Any column created *after* that date, or derived from the answer, is a leak. Three ways I check each column: (1) does its meaning refer to the exit? (2) is it blank for exactly the people who stayed? (3) does it predict the answer almost perfectly on its own?

In [16]:
from sklearn.metrics import roc_auc_score
# (2) blank pattern vs label, (3) single-column predictive power (AUC; 0.5 = useless, 1.0 = perfect)
rows = []
for c in LEAKAGE_COLUMNS:
    present = d[c].notna().astype(int)
    rows.append({"column": c, "filled_for_leavers_%": round(100 * present[d[TARGET] == 1].mean(), 1),
                 "filled_for_stayers_%": round(100 * present[d[TARGET] == 0].mean(), 1),
                 "AUC_of_'is it filled?'": round(roc_auc_score(d[TARGET], present), 3)})
# voluntary flag has no blanks: use its value
rows[-1].update({"filled_for_leavers_%": 100.0, "filled_for_stayers_%": 100.0, "AUC_of_'is it filled?'": round(roc_auc_score(d[TARGET], d.voluntary_left_within_12m), 3)})
leak = pd.DataFrame(rows)
display(leak)

# Same test on every legitimate numeric feature, to see what an HONEST column looks like
num = [c for c in d.select_dtypes("number").columns if c not in LEAKAGE_COLUMNS + [TARGET, "row_id"]]
auc = {c: roc_auc_score(d[TARGET], d[c].fillna(d[c].median())) for c in num}
auc = pd.Series({c: max(a, 1 - a) for c, a in auc.items()}).sort_values(ascending=False)
F["top_clean_auc"] = auc.head(5).round(3).to_dict()
print("Strongest LEGITIMATE single columns (AUC):"); print(auc.head(5).round(3).to_string())
print("\nChecks on provenance columns:  source AUC =", round(roc_auc_score(d[TARGET], (d.source == 'historical_extension').astype(int)), 3),
      "| observation_date AUC =", round(roc_auc_score(d[TARGET], d.observation_date.astype('int64')), 3))
F["leak_table"] = leak.to_dict("records")

,column,filled_for_leavers_%,filled_for_stayers_%,AUC_of_'is it filled?'
0,exit_date,100.0,0.0,1.00
1,resignation_date,92.0,0.0,0.96
2,exit_reason,100.0,0.0,1.00
3,exit_interview_done,100.0,0.0,1.00
4,fnf_settlement_amount,100.0,0.0,1.00
5,voluntary_left_within_12m,100.0,100.0,0.96


Strongest LEGITIMATE single columns (AUC):
last_engagement_score      0.612
months_since_last_raise    0.599
manager_changes_12m        0.572
comp_ratio                 0.553
wfh_share_3m               0.552

Checks on provenance columns:  source AUC = 0.51 | observation_date AUC = 0.497


In [17]:
# Prove it: the same simple model, with and without the leaky columns.  (A demonstration only - the real pipeline is Task 2.)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cats = ["department_id", "role_id", "work_location", "work_mode", "employment_type", "highest_education"]
X = d.drop(columns=[TARGET] + LEAKAGE_COLUMNS + NON_FEATURE_COLUMNS).copy()
X["work_location"] = X.work_location.map(normalise_location)
X_leaky = X.copy()
X_leaky["fnf_settlement_amount"] = d.fnf_settlement_amount.fillna(0)
X_leaky["exit_interview_done"] = d.exit_interview_done.fillna("none")
X_leaky["exit_reason"] = d.exit_reason.fillna("none")

def make_model(X):
    cat_cols = [c for c in X.columns if X[c].dtype == object or str(X[c].dtype).startswith("str") or c in cats]
    num_cols = [c for c in X.columns if c not in cat_cols]
    pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
                             ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"),
                                                   OneHotEncoder(handle_unknown="ignore")), cat_cols)])
    return make_pipeline(pre, LogisticRegression(max_iter=2000, class_weight="balanced"))

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)     # stratified by label, grouped by employee
res = {}
for name, XX in {"Clean features": X, "Clean + exit columns (LEAKY)": X_leaky}.items():
    sc = cross_validate(make_model(XX), XX, d[TARGET], groups=d.employee_id, cv=cv, scoring=["roc_auc", "average_precision"])
    res[name] = {"roc_auc": sc["test_roc_auc"].mean(), "roc_auc_sd": sc["test_roc_auc"].std(), "pr_auc": sc["test_average_precision"].mean()}
res["Dummy ('nobody leaves')"] = {"roc_auc": 0.5, "roc_auc_sd": 0.0, "pr_auc": y.mean()}
leak_demo = pd.DataFrame(res).T.round(3); display(leak_demo)
F["leak_demo"] = leak_demo.reset_index().rename(columns={"index": "model"}).to_dict("records")
ref = pd.read_excel(WORKBOOK, "baseline_benchmarks")[["model", "roc_auc_mean", "pr_auc_mean"]]
print("Reference benchmarks supplied in the workbook:"); display(ref)

,roc_auc,roc_auc_sd,pr_auc
Clean features,0.697,0.028,0.299
Clean + exit columns (LEAKY),1.000,0.000,1.000
Dummy ('nobody leaves'),0.500,0.000,0.146


Reference benchmarks supplied in the workbook:


,model,roc_auc_mean,pr_auc_mean
0,Dummy (always stays),0.500,0.146
1,Logistic Regression (class_weight=balanced),0.713,0.332
2,"Decision Tree (depth 4, balanced)",0.650,0.256
3,"Random Forest (300 trees, balanced)",0.715,0.336
4,LEAKY Logistic Regression (adds exit information),1.000,1.000


In [18]:
fig, ax = plt.subplots(figsize=(6.4, 3.0))
names = ["Dummy\n('nobody leaves')", "Honest model\n(clean features)", "Leaky model\n(+ exit columns)"]
vals = [0.5, res["Clean features"]["roc_auc"], res["Clean + exit columns (LEAKY)"]["roc_auc"]]
bars = ax.bar(names, vals, color=[GREY, BLUE, RED], width=.55)
for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width()/2, v + .02, f"{v:.2f}", ha="center", fontweight="bold")
ax.axhline(.5, color="k", lw=.8, ls=":"); ax.set_ylim(0, 1.12); ax.set_ylabel("ROC-AUC (5-fold CV)")
ax.set_title("Too good to be true: leakage gives a perfect score"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.savefig(FIGURES / "fig4_leakage.png"); plt.show()

In [19]:
# Leakage can also hide in OTHER sheets that Task 2 will join.  Do they contain information from after the snapshot?
snap = pd.Timestamp("2025-09-30")
surv = pd.read_excel(WORKBOOK, "engagement_survey", parse_dates=["survey_date"])
perf = pd.read_excel(WORKBOOK, "performance_reviews", parse_dates=["appraisal_date"])
att = pd.read_excel(WORKBOOK, "attendance_monthly_features", parse_dates=["month"])
scoring = pd.read_excel(WORKBOOK, "attrition_scoring_2026", nrows=3)
future = pd.DataFrame({"sheet": ["engagement_survey", "performance_reviews", "attendance_monthly_features"],
                       "rows": [len(surv), len(perf), len(att)],
                       "rows_after_30_Sep_2025": [int((surv.survey_date > snap).sum()), int((perf.appraisal_date > snap).sum()), int((att.month > snap).sum())],
                       "latest_date": [surv.survey_date.max().date(), perf.appraisal_date.max().date(), att.month.max().date()]})
future["pct_after_snapshot"] = (100 * future.rows_after_30_Sep_2025 / future.rows).round(1)
display(future)
F["future_sheets"] = future.astype({"latest_date": str}).to_dict("records")
missing_in_scoring = [c for c in df.columns if c not in scoring.columns]
print("Columns present in training table but ABSENT from the 2026 scoring table:", missing_in_scoring)
pa = pd.read_excel(WORKBOOK, "protected_attributes")
print("Gender/age live in a separate sheet:", list(pa.columns), "-> not in the attrition table (good).")

,sheet,rows,rows_after_30_Sep_2025,latest_date,pct_after_snapshot
0,engagement_survey,2829,595,2026-06-15,21.0
1,performance_reviews,1740,364,2026-04-01,20.9
2,attendance_monthly_features,7339,4985,2026-09-01,67.9


Columns present in training table but ABSENT from the 2026 scoring table: ['left_within_12m', 'voluntary_left_within_12m', 'source', 'exit_date', 'resignation_date', 'exit_reason', 'exit_interview_done', 'fnf_settlement_amount']


Gender/age live in a separate sheet: ['person_ref', 'record_type', 'gender', 'age_years', 'age_band'] -> not in the attrition table (good).


**Findings — columns that leak the answer (6):**

| Column | Why it leaks | Evidence |
|---|---|---|
| `exit_date` | The last working day exists only for people who left | filled for 100% of leavers, 0% of stayers → AUC 1.0 |
| `exit_reason` | Recorded at exit | same pattern, AUC 1.0 |
| `exit_interview_done` | Held at exit | same pattern, AUC 1.0 |
| `fnf_settlement_amount` | Paid at exit | same pattern, AUC 1.0 |
| `resignation_date` | Exists only for resigners | filled for 92% of leavers, 0% of stayers → AUC 0.96 |
| `voluntary_left_within_12m` | A sub-label of the target itself | 321 of 349 leavers = 1; stayers are always 0 → AUC 0.96 |

**Proof:** adding the exit columns to an otherwise ordinary model lifts ROC-AUC from ≈0.7 to **1.00**, exactly the "too good to be true" warning that the reference benchmark sheet describes. The best legitimate single column (engagement) manages only ≈0.61. When a feature, or a whole model, is *much* better than that, suspect leakage.

**Not leakage, but not a feature either:** `row_id`, `employee_id` (use for grouping only), `source` (tells the model which simulation produced the row), and `observation_date` (use for time-aware splitting only).

**Unconfirmed, not assumed safe:** `dept_attrition_rate_prior_12m` is *documented* as looking at the 12 months before the snapshot, but I have not been able to confirm it. Section 12 shows evidence that is consistent with that, yet it is not proof, so it is open item O2. (`comp_ratio`, `months_since_last_raise` and the attendance windows are documented as ending on or before the snapshot date.)

**A trap waiting in Task 2:** the survey, appraisal and attendance sheets run past the snapshot (up to mid-2026). If those are aggregated into features **without cutting off at `observation_date`**, future information leaks in through the back door. The 2026 scoring table has no exit columns, which confirms that the model must not use them: they will not exist at prediction time.

## 9. Check against the planted-problem answer key
The workbook ships an answer key (`answer_key_planted_dirt`). Every check above is **computed from the data itself**; nothing is copied from the key. I use the key here only to validate how complete the checks were. *Honest note: the key sheet sits in the same workbook and was opened while first exploring the file, so this is a validation, not a blind test. For a true blind test, do your own pass first and compare afterwards.*

In [20]:
key = pd.read_excel(WORKBOOK, "answer_key_planted_dirt")
dup_found = set(F["dup_employee_ids"])
mine = {
    "DUPLICATE_ROW":       dup_found,
    "NEGATIVE_TENURE":     set(df.loc[df.tenure_months < 0, "employee_id"]),
    "SALARY_TYPO_OUTLIER": set(d.loc[d.mid_vs_role_median > 3, "employee_id"]),
    "IMPOSSIBLE_VALUE":    set(df.loc[df.attendance_rate_3m > 1, "employee_id"]),
}
loc_found = df.loc[df.work_location != loc_clean, ["employee_id", "work_location"]]
cmp_rows = []
for t, found in mine.items():
    planted = set(key.loc[key.issue_type == t, "employee_id"])
    cmp_rows.append({"issue_type": t, "planted": len(planted), "found_by_audit": len(found & planted),
                     "extra_flags": len(found - planted), "missed": len(planted - found)})
planted_loc = set(key.loc[key.issue_type == "INCONSISTENT_CATEGORY", "employee_id"])
cmp_rows.append({"issue_type": "INCONSISTENT_CATEGORY", "planted": len(planted_loc),
                 "found_by_audit": len(set(loc_found.employee_id) & planted_loc),
                 "extra_flags": len(set(loc_found.employee_id) - planted_loc), "missed": len(planted_loc - set(loc_found.employee_id))})
mv = key[key.issue_type == "MISSING_VALUE"].detail.str.extract(r"^(\w+)")[0].value_counts()
cmp_rows.append({"issue_type": "MISSING_VALUE (commute + education)", "planted": int(mv.sum()),
                 "found_by_audit": int(df.commute_distance_km.isna().sum() + df.highest_education.isna().sum()),
                 "extra_flags": 0, "missed": 0})
cmp = pd.DataFrame(cmp_rows); cmp.to_csv(TABLES / "answer_key_comparison.csv", index=False)
F["answer_key"] = cmp.to_dict("records")
display(cmp)
print("Planted missing values by column in the key:", mv.to_dict())
print("\nProblems my audit found that the answer key does NOT list:")
extras = {"wfh_share_3m outside 0-1": int(rules["wfh_share_3m < 0"].sum() + rules["wfh_share_3m > 1"].sum()),
          "total_experience_years < tenure": int(rules["total_experience < tenure (by more than 0.1 yr)"].sum()),
          "total_experience_years < 0": int(rules["total_experience_years < 0"].sum()),
          "avg_hours_worked_3m = 1.8 (implausible)": 1,
          "post-snapshot rows in survey/appraisal/attendance sheets": int(future.rows_after_30_Sep_2025.sum())}
F["extras_beyond_key"] = extras; print(extras)

,issue_type,planted,found_by_audit,extra_flags,missed
0,DUPLICATE_ROW,6,6,0,0
1,NEGATIVE_TENURE,4,4,0,0
2,SALARY_TYPO_OUTLIER,3,3,0,0
3,IMPOSSIBLE_VALUE,2,2,0,0
4,INCONSISTENT_CATEGORY,13,13,0,0
5,MISSING_VALUE (commute + education),70,70,0,0


Planted missing values by column in the key: {'commute_distance_km': 55, 'highest_education': 15}

Problems my audit found that the answer key does NOT list:
{'wfh_share_3m outside 0-1': 213, 'total_experience_years < tenure': 137, 'total_experience_years < 0': 10, 'avg_hours_worked_3m = 1.8 (implausible)': 1, 'post-snapshot rows in survey/appraisal/attendance sheets': 5944}


**Result:** every planted problem category was caught (duplicates, negative tenure, salary typos, attendance > 1, spelling variants, and the 70 random blanks in commute/education). The audit also surfaced extra issues the key does not list. The "planted" list covers only injected defects; the other findings come from *structural* checks (does the blank have a reason? does experience ≥ tenure?) that any real dataset needs.

## 10. Automatic profile (ydata-profiling) vs my manual audit
The automatic tool produces an HTML report in one line. I run it on the **raw** table to see what it catches and what only a human catches.

In [21]:
prof_status, alert_summary = "not run", {}
try:
    from collections import defaultdict
    from ydata_profiling import ProfileReport
    cfg = dict(minimal=False, progress_bar=False, interactions=None,           # skip the slow pairwise-scatter section
               correlations={k: {"calculate": k == "pearson"} for k in ["auto", "pearson", "spearman", "kendall", "phi_k", "cramers"]},
               missing_diagrams={"bar": False, "matrix": False, "heatmap": False})
    profile = ProfileReport(df_raw.drop(columns=["row_id"]), title="CortexHR attrition_dataset - automatic profile", **cfg)
    profile.to_file(REPORTS / "ydata_profile_attrition.html")
    grouped = defaultdict(list)
    for a in profile.get_description().alerts:
        grouped[a.alert_type.name].append(a.column_name)
    alert_summary = {k: v for k, v in grouped.items()}
    prof_status = "ok"
except Exception as e:                       # profile is a nice-to-have, never block the audit on it
    prof_status = f"skipped ({type(e).__name__}: {e})"
F["ydata"] = {"status": prof_status, "alerts": alert_summary}
print(prof_status)
for k, v in alert_summary.items(): print(f"{k:17s}", v)

skipped (ModuleNotFoundError: No module named 'pkg_resources')


**Comparison: what the automatic report does and does not catch.**

| Check | Automatic profile (ydata-profiling) | My manual audit |
|---|---|---|
| Missing values | ✅ flags 10 columns, but not `highest_education` (15 blanks); gives a count, never the *reason* | ✅ plus a cause and a plan for each |
| Duplicate rows | ✅ finds the 6 duplicates | ✅ |
| Zero-heavy columns | ✅ 9 columns flagged | ✅ (overtime: 81% zeros) |
| Class imbalance of the **label** | ❌ flags `employment_type`, `internal_transfers`, `manager_changes_12m` but *not* `left_within_12m` (14.6% is under its threshold) | ✅ |
| Data leakage | ⚠️ its high-correlation list *includes* `left_within_12m`, `voluntary_left_within_12m` and `fnf_settlement_amount` — a hint, but no explanation of why | ✅ meaning + blank pattern + AUC test + model proof |
| Impossible values (tenure < 0, experience < tenure, attendance > 1) | ❌ shows min/max only; does not know the rules | ✅ rules from the dictionary |
| 10× salary typo | ❌ no alert | ✅ pay ÷ comp_ratio vs role mid-point |
| Outliers | ❌ no outlier alert, only histograms | ✅ IQR vs z-score + business rule |
| Messy city spellings | ❌ lists the values, raises no alert | ✅ |

**Takeaway:** the automatic profile is a fast, cheap first scan and a good way to double-check my counts. But everything that needs *business knowledge* — why a value is blank, whether a number is possible, whether a column exists at prediction time — needs a person who understands the question being asked.

## 11. Decision log
Every issue has a written decision or, where the answer is not ours to give, a documented open item (D03, D24; section 12). These become the rules of the reusable pipeline in **Task 2** (the `Task 2 rule` column is the hand-off).

In [22]:
decisions = [
 # id, area, issue, rows, decision, why, task2 rule
 ("D01","Duplicates","6 exact duplicate rows (employee appears twice)",6,"DROP second copy","Duplicates inflate scores when copies fall in train and test","dedupe on all columns except row_id; GroupKFold by employee_id"),
 ("D02","Consistency","13 city labels spelled differently (Bangalore, BENGALURU, Madras, Delhi-NCR ...)",13,"FIX with lookup table","Same city must be one category","normalise_location() in src/cortexhr/audit.py"),
 ("D03","Invalid","Negative tenure (obs. date before join date): EMP-0248, 0280, 0568, 0596",4,"HOLD - open item O1. Raw value left unchanged; add tenure_was_invalid flag; until the team answers, EXCLUDE these 4 rows from model training (documented, not imputed)","True join dates are unknown; any fix would be a guess","flag tenure<0; drop flagged rows from training until O1 is closed, then apply the team's answer"),
 ("D04","Invalid","Attendance rate above 100% (1.12)",2,"CAP at 1.0 + FLAG","Cannot exceed 100%","clip(0,1)"),
 ("D05","Invalid","wfh_share_3m outside 0-1",213,"CLIP to 0-1","Small noise (|x|<0.1); a share must lie in 0-1","clip(0,1)"),
 ("D06","Invalid","Total experience below tenure (137 rows) or negative (10 rows), all in historical rows",137,"FLOOR at tenure in years","Total experience includes time at this company","total_exp=max(total_exp, tenure_months/12)"),
 ("D07","Outlier","Salary typos: 3 values 10x too high (EMP-0034, 0433, 0601)",3,"CORRECT (divide by 10)","pay/comp_ratio disagrees with role mid-point by ~11x; /10 fits the level","if pay/comp_ratio > 3x role median: pay/=10"),
 ("D08","Outlier","Genuine high salaries (IQR flags 93, z-score 26)",93,"KEEP","Senior staff are paid more; this is real","log1p(annual_fixed_pay) for linear models"),
 ("D09","Outlier","Daily hours 1.8 / attendance 20% (EMP-0313)",1,"KEEP + FLAG for HR review","Two columns agree, so looks like a real long absence","RobustScaler / tree models"),
 ("D10","Outlier","Overtime heavy tail (81% zeros, max 50 h); tenure up to 164 m",None,"KEEP","Plausible values; IQR over-flags zero-heavy data","log1p(overtime)"),
 ("D11","Missing","months_since_last_promotion 61% blank = never promoted",1459,"KEEP; FILL with tenure_months; add never_promoted flag","Blank carries meaning; dropping loses signal","fill=tenure_months; flag"),
 ("D12","Missing","last_raise_pct 8.5% blank = never had a raise",202,"KEEP; FILL 0; add never_raised flag","Structural blank","fill=0; flag"),
 ("D13","Missing","last_performance_rating 7.6% blank = joined <12 months ago",182,"KEEP; FILL median; add no_rating_yet flag","Structural blank","median from TRAIN only; flag"),
 ("D14","Missing","last_engagement_score 22% blank = no survey response",528,"KEEP; FILL median; add engagement_missing flag","Non-response may carry signal (16.7% vs 14.0% attrition, not significant)","median by department from TRAIN only; flag"),
 ("D15","Missing","commute_distance_km 2.3% blank",55,"FILL median by city; add flag","Random gap, small","median by city from TRAIN only; flag"),
 ("D16","Missing","highest_education 0.6% blank",15,"FILL 'Unknown'","Random gap, categorical","constant 'Unknown'"),
 ("D17","Imbalance","14.6% left, 85.4% stayed (5.8 : 1)",349,"KEEP the data as is; no copying rows","Real-world rate; oversampling before splitting leaks","StratifiedGroupKFold; class_weight='balanced'; metrics = ROC-AUC, PR-AUC, recall@top-10%"),
 ("D18","Leakage","exit_date, resignation_date, exit_reason, exit_interview_done, fnf_settlement_amount (post-outcome)",None,"DROP before any modelling","Exist only after the outcome; give AUC 1.0","assert none of LEAKAGE_COLUMNS in feature matrix"),
 ("D19","Leakage","voluntary_left_within_12m (sub-label of the target)",None,"DROP","Derived from the answer","same assertion"),
 ("D20","Leakage","Survey, appraisal and attendance sheets include data after 30 Sep 2025",None,"CUT OFF at observation_date when building features","Future information would leak in","filter date <= observation_date"),
 ("D21","Non-features","row_id, employee_id, source, observation_date",None,"EXCLUDE from features (employee_id for grouping; observation_date for time split)","Identifiers/provenance, not employee traits","NON_FEATURE_COLUMNS"),
 ("D22","Fairness","gender and age are held in a separate sheet",None,"KEEP OUT of the feature set; use only to test fairness (Task 17)","Avoid direct discrimination; still need them to test for bias","never join protected_attributes into X"),
 ("D24","Leakage (unconfirmed)","dept_attrition_rate_prior_12m: cut-off date not confirmed",None,"HOLD - open item O2. Keep as a CANDIDATE feature; train every model with and without it and report both until confirmed","If it used exits after the observation date it would leak the answer; the evidence so far is only consistent with a backward-looking calculation","two feature sets (with/without); drop it if the team says it uses post-observation exits"),
 ("D23","Process","Imputation, scaling and encoding must not see the test data",None,"FIT on training fold only (sklearn Pipeline)","Otherwise test information leaks in","all transforms inside Pipeline/ColumnTransformer"),
]
dec = pd.DataFrame(decisions, columns=["id", "area", "issue", "rows", "decision", "why", "task2_rule"])
dec.to_csv(TABLES / "decision_log.csv", index=False)
F["decisions"] = dec.astype(object).where(dec.notna(), None).to_dict("records")   # None -> JSON null (NaN is not valid JSON)

md_lines = ["# Decision log — Task 1 audit", "",
            "Each row is a rule the Task 2 pipeline must implement (D03 and D24 are on HOLD pending the Python team). Rows = how many records are affected.", "",
            "| ID | Area | Issue | Rows | Decision | Why | Task 2 rule |", "|---|---|---|---|---|---|---|"]
for r in dec.itertuples():
    md_lines.append(f"| {r.id} | {r.area} | {r.issue} | {'' if pd.isna(r.rows) else int(r.rows)} | {r.decision} | {r.why} | {r.task2_rule} |")
(DOCS / "decisions.md").write_text("\n".join(md_lines) + "\n")
dec[["id", "area", "issue", "decision"]]

,id,area,issue,decision
0,D01,Duplicates,6 exact duplicate rows (employee appears twice),DROP second copy
1,D02,Consistency,"13 city labels spelled differently (Bangalore, BENGALURU, Madras, Delhi-NCR ...",FIX with lookup table
2,D03,Invalid,"Negative tenure (obs. date before join date): EMP-0248, 0280, 0568, 0596",HOLD - open item O1. Raw value left unchanged; add tenure_was_invalid flag; ...
3,D04,Invalid,Attendance rate above 100% (1.12),CAP at 1.0 + FLAG
4,D05,Invalid,wfh_share_3m outside 0-1,CLIP to 0-1
5,D06,Invalid,"Total experience below tenure (137 rows) or negative (10 rows), all in histo...",FLOOR at tenure in years
6,D07,Outlier,"Salary typos: 3 values 10x too high (EMP-0034, 0433, 0601)",CORRECT (divide by 10)
7,D08,Outlier,"Genuine high salaries (IQR flags 93, z-score 26)",KEEP
8,D09,Outlier,Daily hours 1.8 / attendance 20% (EMP-0313),KEEP + FLAG for HR review
9,D10,Outlier,"Overtime heavy tail (81% zeros, max 50 h); tenure up to 164 m",KEEP


## 12. Open items for the Python team
Two questions can only be answered by the people who built the data. **I have not guessed the answers.** Each item states what is known, what I did in the meantime, and has a place to record the team's response. The same table is in `docs/open_items.md` and `reports/tables/open_items.csv`.

In [23]:
# ---- O1: negative tenure -------------------------------------------------------------------
neg = df.loc[df.tenure_months < 0, ["row_id", "employee_id", "observation_date", "tenure_months", "source", TARGET]]
display(neg)
print("Join dates for these 4 employees elsewhere in the workbook: none (workforce_events has no employee ids; salary_training is keyed by hire_id).")

# ---- O2: does dept_attrition_rate_prior_12m look backwards?  Test against the headcount sheet ----
hc = pd.read_excel(WORKBOOK, "monthly_headcount_history", parse_dates=["month"])
obs_real = pd.Timestamp("2025-09-30")
real = d[d.source == "master_employee_snapshot"]
per_dept = real.groupby("department_id").dept_attrition_rate_prior_12m.agg(["nunique", "first"])

def exit_rate(dept, start, end):
    s = hc[(hc.department_id == dept) & (hc.month > start) & (hc.month <= end)]
    return s.exits.sum() / s.headcount_start.mean()

chk = pd.DataFrame({
    "dataset_value": per_dept["first"],
    "exits_in_prior_12m": [exit_rate(k, obs_real - pd.DateOffset(months=12), obs_real) for k in per_dept.index],
    "exits_in_next_12m":  [exit_rate(k, obs_real, obs_real + pd.DateOffset(months=12)) for k in per_dept.index]}).round(4)
chk["abs_diff_vs_prior"] = (chk.dataset_value - chk.exits_in_prior_12m).abs().round(4)
chk["abs_diff_vs_next"] = (chk.dataset_value - chk.exits_in_next_12m).abs().round(4)
display(chk)
mae_prior, mae_next = chk.abs_diff_vs_prior.mean(), chk.abs_diff_vs_next.mean()
const = bool((per_dept["nunique"] == 1).all())
F["dept_rate_check"] = {"constant_within_dept_real_rows": const, "mean_abs_diff_prior_pts": round(100 * mae_prior, 1),
                        "mean_abs_diff_next_pts": round(100 * mae_next, 1), "table": chk.reset_index().to_dict("records")}
print(f"Real rows: value constant within each department = {const}.  Mean gap to PRIOR-12m exits: {100*mae_prior:.1f} pts;  to NEXT-12m exits: {100*mae_next:.1f} pts.")
print("Historical rows (2,000) were NOT tested.")

,row_id,employee_id,observation_date,tenure_months,source,left_within_12m
108,109,EMP-0248,2025-09-30,-4,master_employee_snapshot,0
125,126,EMP-0280,2025-09-30,-5,master_employee_snapshot,0
312,313,EMP-0568,2025-09-30,-2,master_employee_snapshot,0
339,340,EMP-0596,2025-09-30,-1,master_employee_snapshot,0


Join dates for these 4 employees elsewhere in the workbook: none (workforce_events has no employee ids; salary_training is keyed by hire_id).


,dataset_value,exits_in_prior_12m,exits_in_next_12m,abs_diff_vs_prior,abs_diff_vs_next
department_id,,,,,
D01,0.1448,0.1469,0.1366,0.0021,0.0082
D02,0.0000,0.0000,0.2368,0.0000,0.2368
D03,0.0556,0.0632,0.1419,0.0076,0.0863
D04,0.2051,0.2069,0.1387,0.0018,0.0664
D05,0.1667,0.1920,0.0000,0.0253,0.1667
D06,0.2826,0.2694,0.0573,0.0132,0.2253
D07,0.0000,0.0000,0.1765,0.0000,0.1765
D08,0.2353,0.2412,0.1846,0.0059,0.0507
D09,0.2069,0.2017,0.2658,0.0052,0.0589


Real rows: value constant within each department = True.  Mean gap to PRIOR-12m exits: 0.7 pts;  to NEXT-12m exits: 10.8 pts.
Historical rows (2,000) were NOT tested.


In [24]:
ev2 = (f"For the 383 real rows the value is {'constant' if const else 'not constant'} within each department and is on average {100*mae_prior:.1f} points from the "
       f"exits-in-the-prior-12-months rate computed from monthly_headcount_history, versus {100*mae_next:.1f} points from the following 12 months. "
       "This is consistent with a backward-looking calculation but is NOT proof; the 2,000 historical rows were not tested.")
open_items = pd.DataFrame([
 {"id": "O1", "question": "Negative tenure: what are the correct joining dates for EMP-0248, EMP-0280, EMP-0568 and EMP-0596?",
  "why_it_matters": "Tenure is negative (-4, -5, -2, -1 months), which is impossible. Without the join dates any fix is a guess.",
  "evidence_so_far": "All 4 are real employees observed on 30 Sep 2025, all label = 0. No join dates exist elsewhere in the workbook.",
  "handling_meanwhile": "Raw values untouched. A tenure_was_invalid flag will be added in Task 2. The 4 rows are excluded from model training until answered (documented, not imputed).",
  "owner": "Python team", "status": "OPEN", "team_response": "(pending - record here)", "date_answered": "", "final_action": ""},
 {"id": "O2", "question": "Is dept_attrition_rate_prior_12m calculated ONLY from employee exits that occurred before each row's observation date?",
  "why_it_matters": "If it includes exits after the observation date it leaks the answer (the department's own future exits).",
  "evidence_so_far": ev2,
  "handling_meanwhile": "Kept as a candidate feature, not assumed safe. Every model is trained and reported with AND without it until answered.",
  "owner": "Python team", "status": "OPEN", "team_response": "(pending - record here)", "date_answered": "", "final_action": ""},
 {"id": "O3", "question": "(Additional observation) Why do only the 2,000 simulated rows have total experience below tenure (137 rows)?",
  "why_it_matters": "Suggests the two sources were generated by different logic; matters if the pipeline treats them as one population.",
  "evidence_so_far": "0 such rows among the 383 real employees; 137 among simulated rows.",
  "handling_meanwhile": "Rule D06 (floor experience at tenure) applied and documented; revisit if the team explains the cause.",
  "owner": "Python team", "status": "OPEN (lower priority)", "team_response": "(pending - record here)", "date_answered": "", "final_action": ""},
])
open_items.to_csv(TABLES / "open_items.csv", index=False)
F["open_items"] = open_items.to_dict("records")

lines = ["# Open items for the Python team", "",
         "Nothing below has been assumed or silently corrected. Fill in *Team response*, *Date answered* and *Final action* when the team replies,",
         "then change Status to CLOSED and update the matching decision (D03 for O1, D24 for O2) in `decisions.md`.", ""]
for r in open_items.itertuples():
    lines += [f"## {r.id} - {r.status}", "", f"**Question:** {r.question}", "", f"**Why it matters:** {r.why_it_matters}", "",
              f"**Evidence so far:** {r.evidence_so_far}", "", f"**Handling meanwhile:** {r.handling_meanwhile}", "",
              f"**Owner:** {r.owner}", "", "| Team response | Date answered | Final action |", "|---|---|---|", "| (pending) | | |", ""]
(DOCS / "open_items.md").write_text("\n".join(lines))
open_items[["id", "status", "question", "handling_meanwhile"]]

,id,status,question,handling_meanwhile
0,O1,OPEN,"Negative tenure: what are the correct joining dates for EMP-0248, EMP-0280, ...",Raw values untouched. A tenure_was_invalid flag will be added in Task 2. The...
1,O2,OPEN,Is dept_attrition_rate_prior_12m calculated ONLY from employee exits that oc...,"Kept as a candidate feature, not assumed safe. Every model is trained and re..."
2,O3,OPEN (lower priority),"(Additional observation) Why do only the 2,000 simulated rows have total exp...",Rule D06 (floor experience at tenure) applied and documented; revisit if the...


In [25]:
(REPORTS / "findings.json").write_text(json.dumps(F, indent=2, default=str, allow_nan=False))   # fail loudly if NaN sneaks in
print("Saved findings.json with", len(F), "sections;  figures:", sorted(p.name for p in FIGURES.glob('*.png')))

Saved findings.json with 33 sections;  figures: ['fig1_missing.png', 'fig2_outliers.png', 'fig3_imbalance.png', 'fig4_leakage.png']


## Summary
* **Shape:** 2,389 rows × 42 columns (383 real employees + 2,000 simulated history rows + 6 duplicates).
* **Missing:** mostly *structural* — blanks mean "never promoted / no raise / no appraisal yet". Only commute (55) and education (15) are random gaps.
* **Impossible values:** negative tenure (4), attendance > 100% (2), 13 spelling variants, and extras the answer key does not list.
* **Outliers:** IQR flags 1.6–7× more rows than z-score; 3 salaries are 10× typos; everything else is real.
* **Imbalance:** 14.6% leave. Use stratified splits and PR-AUC/recall, not accuracy.
* **Leakage:** 6 columns reveal the answer (AUC ≈ 1.0). Drop them; also cut off other sheets at the snapshot date.
* **Open items:** 2 questions (negative tenure; department attrition-rate cut-off) await the Python team. Nothing was assumed. Responses are recorded in `docs/open_items.md`.
* **Next:** Task 2 turns the decision log into a reusable pipeline.